# Predicting the Sale Price of Bulldozers using Machine Learning

## 1.Problem definition

> How well can we predict the future sale price of a bulldozer, given its characteristics previous examples of how much similar bulldozers have been sold for?

## 2.Data

The data is downloaded from the Kaggle Bluebook for Bulldozers competition:
https://www.kaggle.com/competitions/bluebook-for-bulldozers/data

**_ Note: Place the files in a `data/` folder at the root of the project. _**

There are 3 main datasets:

- Train.csv is the training set, which contains data through the end of 2011.
- Valid.csv is the validation set, which contains data from January 1, 2012 - April 30, 2012 You make predictions on this set throughout the majority of the competition. Your score on this set is used to create the public leaderboard.
- Test.csv is the test set, which won't be released until the last week of the competition. It contains data from May 1, 2012 - November 2012. Your score on the test set determines your final rank for the competition.

## 3.Evaluation

The evaluation metric for this competition is the RMSLE (root mean squared log error) between the actual and predicted auction prices.

## 4.Features

Kaggle provides a data dictionary detailing all of te features of the dataset. You can view this data dictionary on the file:

```
bulldozer-price-prediction/
└── data/
    └── Data Dictionary.xlsx   # This file
```


In [ ]:
# Mount Google Drive to access files in Colab
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

# Plot appear inside the notebook
%matplotlib inline

In [ ]:
# Importing training and validation datasets
# df = pd.read_csv("data/TrainAndValid.csv", low_memory=False)
df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/bulldozer/TrainAndValid.csv",
    low_memory=False,
)

df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["saledate"][:1000], df["SalePrice"][:1000])
plt.show()

In [ ]:
df.saledate[:1000]

In [ ]:
df.saledate.dtypes

In [ ]:
df.SalePrice.plot.hist()

In [ ]:
# Parsing dates
# df = pd.read_csv("data/TrainAndValid.csv", low_memory=False, parse_dates=["saledate"])
df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/bulldozer/TrainAndValid.csv",
    low_memory=False,
    parse_dates=["saledate"],
)

In [ ]:
df.saledate.dtypes

In [ ]:
df.saledate[:1000]

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["saledate"][:1000], df["SalePrice"][:1000])
plt.show()

In [ ]:
df.head()

In [ ]:
df.head().T

In [ ]:
# sorting by saledate in date order
df.sort_values(by=["saledate"], inplace=True)
df.saledate.head(20)

In [ ]:
# make a copy
df_temp = df.copy()

In [ ]:
df_temp.saledate.head(20)

In [ ]:
df_temp["saleYear"] = df_temp["saledate"].dt.year
df_temp["saleMonth"] = df_temp["saledate"].dt.month
df_temp["saleDay"] = df_temp["saledate"].dt.day
df_temp["saleDayOfWeek"] = df_temp["saledate"].dt.day_of_week
df_temp["saleDayOfYear"] = df_temp["saledate"].dt.day_of_year

In [ ]:
df_temp.drop("saledate", axis=1, inplace=True)

### Convert string to categories


In [ ]:
df_temp.head().T

In [ ]:
str_columns = df_temp.select_dtypes(include=["object", "string"]).columns

for col in str_columns:
    df_temp[col] = df_temp[col].astype("category").cat.as_ordered()

df_temp.info()

In [ ]:
df_temp["state"].cat.categories

In [ ]:
df_temp["state"].cat.codes

### Check missing data


In [ ]:
# Export current temp dataframe
# df_temp.to_csv("data/train_tmp.csv", index=False)
df_temp.to_parquet(
    "/content/drive/MyDrive/Colab Notebooks/bulldozer/train_tmp.parquet", index=False
)

# import data tmp
# df_tmp = pd.read_csv("data/train_tmp.csv", low_memory=False)
df_tmp = pd.read_parquet(
    "/content/drive/MyDrive/Colab Notebooks/bulldozer/train_tmp.parquet"
)
df_tmp.head().T

In [ ]:
df_tmp.info()

#### fill numeric values


In [ ]:
numeric_columns = df_tmp.select_dtypes(include=["number"]).columns

for column in numeric_columns:
    missing_mask = df_tmp[column].isnull()
    if missing_mask.any():
        df_tmp[column + "_is_missing"] = missing_mask
        df_tmp[column] = df_tmp[column].fillna(df_tmp[column].median())

for column in numeric_columns:
    if df_tmp[column].isna().sum():
        print(f"{column} still has missing values")


In [ ]:
df_tmp["auctioneerID_is_missing"].value_counts()

#### fill categorical values


In [ ]:
df_temp["UsageBand"].dtype

In [ ]:
print(df_tmp["UsageBand"].cat.codes)

In [ ]:
category_columns = df_tmp.select_dtypes(include=["category"]).columns

for column in category_columns:
    missing_mask = df_tmp[column].isnull()
    if missing_mask.any():
        df_tmp[column + "_is_missing"] = missing_mask
    df_tmp[column] = df_tmp[column].cat.codes + 1

for column in category_columns:
    if df_tmp[column].isna().sum():
        print(f"{column} still has missing values")

In [ ]:
df_tmp.select_dtypes(include=["category"])

In [ ]:
df_tmp.info()

In [ ]:
df_tmp.isna().sum()

# Modeling


In [ ]:
%%time

model = RandomForestRegressor(n_jobs=-1, random_state=42)
model.fit(df_tmp.drop("SalePrice", axis=1), df_tmp["SalePrice"])

In [ ]:
df_tmp["saleYear"].value_counts()

In [ ]:
# Split data into training and validation
df_val = df_tmp[df_tmp["saleYear"] == 2012]
df_train = df_tmp[df_tmp["saleYear"] != 2012]
len(df_val), len(df_train)

In [ ]:
# Split data into X & y
X_train, y_train = df_train.drop("SalePrice", axis=1), df_train["SalePrice"]
X_val, y_val = df_val.drop("SalePrice", axis=1), df_val["SalePrice"]

X_train.shape, y_train.shape, X_val.shape, y_val.shape

In [ ]:
# evaluation function
def rmsle(y_test, y_pred):
    """Calculate root mean squared log error between predictions and true labels."""
    return np.sqrt(mean_squared_log_error(y_test, y_pred))


# Evaluate Model
def show_scores(model):
    train_preds = model.predict(X_train)
    val_preds = model.predict(X_val)
    scores = {
        "Training  MAE": mean_absolute_error(y_train, train_preds),
        "Valid MAE": mean_absolute_error(y_val, val_preds),
        "Training RMSLE": rmsle(y_train, train_preds),
        "Valid RMSLE": rmsle(y_val, val_preds),
        "Training R^2": r2_score(y_train, train_preds),
        "Valid R^2": r2_score(y_val, val_preds),
    }
    return scores

#### Testing Model


In [ ]:
%%time

model = RandomForestRegressor(n_jobs=-1, random_state=42, max_samples=10000)
model.fit(X_train, y_train)

In [ ]:
show_scores(model)

#### Hyperparameter tuning with RandomizedSearchCV


In [ ]:
%%time

model = RandomForestRegressor(n_jobs=-1, random_state=42)

rf_grid = {
    "n_estimators": np.arange(10, 100, 10),
    "max_depth": [None, 3, 5, 10],
    "min_samples_split": np.arange(2, 20, 2),
    "min_samples_leaf": np.arange(1, 20, 2),
    "max_features": [0.5, 1, "sqrt", "auto"],
    "max_samples": [10000],
}

rs_model = RandomizedSearchCV(
    estimator=model,
    param_distributions=rf_grid,
    n_iter=100,
    cv=5,
    verbose=True,
)

rs_model.fit(X_train, y_train)

In [ ]:
# the best model
rs_model.best_params_

In [ ]:
# Evaluate
show_scores(rs_model)

**Best hyperparameter**

In [ ]:
%%time

ideal_model = RandomForestRegressor(
    n_estimators=40,
    min_samples_leaf=1,
    min_samples_split=14,
    max_features=0.5,
    n_jobs=-1,
    max_samples=None,
    random_state=42
)

ideal_model.fit(X_train, y_train)

In [ ]:
show_scores(rs_model)